In [10]:
import pandas as pd
from pathlib import Path

# Paths de tus dos archivos (ajústalos si cambian)
FILE1 = Path("S&P 500 Historical Data_2000-2019.csv")
FILE2 = Path("S&P 500 Historical Data2011-2025.csv")


def load_sp500_csv(path: Path) -> pd.DataFrame:
    """Lee y limpia un CSV del S&P 500 con formato tipo Investing.com"""
    
    df = pd.read_csv(path)

    # Normalizar nombres por si difieren entre archivos
    rename_map = {
        "Date": "Date",
        "Price": "Close",
        "Open": "Open",
        "High": "High",
        "Low": "Low",
    }

    df = df.rename(columns=rename_map)

    # Convertir fecha
    df["Date"] = pd.to_datetime(df["Date"])

    # Limpiar números con comas
    for col in ["Close", "Open", "High", "Low"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "")
            .astype(float)
        )

    # Mantener solo columnas necesarias
    df = df[["Date", "Open", "High", "Low", "Close"]]

    return df


def merge_sp500(file1: Path, file2: Path) -> pd.DataFrame:
    """Carga, limpia y combina dos archivos históricos del S&P 500."""
    
    df1 = load_sp500_csv(file1)
    df2 = load_sp500_csv(file2)

    # Unir todo
    df = pd.concat([df1, df2], ignore_index=True)

    # Eliminar duplicados por fecha
    df = df.drop_duplicates(subset="Date")

    # Ordenar cronológicamente
    df = df.sort_values("Date").reset_index(drop=True)

    # Crear target: 1 si la próxima semana baja, 0 si sube
    df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    # remover la última fila que no tiene target
    df = df.iloc[:-1]

    return df


# ==========
# Ejecutar
# ==========
df = merge_sp500(FILE1, FILE2)
print(df.head(5))
print(df.tail(5))

        Date    Open    High     Low   Close  Target
0 2000-01-03  1469.2  1478.0  1438.4  1455.2       0
1 2000-01-04  1455.2  1455.2  1397.4  1399.4       1
2 2000-01-05  1399.4  1413.3  1377.7  1402.1       1
3 2000-01-06  1402.1  1411.9  1392.0  1403.5       1
4 2000-01-07  1403.5  1441.5  1400.5  1441.5       1
           Date     Open     High      Low    Close  Target
6505 2025-11-12  6867.77  6869.91  6829.62  6850.92       0
6506 2025-11-13  6826.47  6828.05  6724.72  6737.49       0
6507 2025-11-14  6672.14  6774.31  6646.87  6734.11       0
6508 2025-11-17  6713.61  6754.50  6638.90  6672.41       0
6509 2025-11-18  6641.19  6666.63  6574.32  6617.37       1


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd


def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    close = df["Close"]

    # MACD (12, 26, 9)
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

    # RSI 14
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs = avg_gain / avg_loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    return df



def make_weekly_dataset(df_daily_ta: pd.DataFrame) -> pd.DataFrame:
    # Ensure date is index
    df = df_daily_ta.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date")

    # Resample to weekly (Friday close)
    weekly = df.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "MACD": "last",
        "MACD_signal": "last",
        "RSI_14": "last",
        # "Volume": "sum",   # if available
    })

    # Weekly % return
    weekly["Weekly_return"] = weekly["Close"].pct_change()

    # Target: did next week go UP?
    weekly["Target"] = (weekly["Close"].shift(-1) > weekly["Close"]).astype(int)

    # Pandemic indicator
    weekly["Pandemic"] = weekly.index.to_series().between(
        "2019-12-01", "2021-12-31"
    ).astype(int)

    # -----------------------------
    #  ⭐ ADD WEEK IDENTIFIERS ⭐
    # -----------------------------

    # Year for each week
    weekly["Year"] = weekly.index.year

    # ISO week number (1–52)
    weekly["Week_Number"] = weekly.index.isocalendar().week.astype(int)

    # Unique identifier (Ex: "2023-47")
    weekly["Week_ID"] = (
        weekly["Year"].astype(str) + "-" + weekly["Week_Number"].astype(str).str.zfill(2)
    )

    # Remove rows with NaN
    weekly = weekly.dropna()

    return weekly



import yfinance as yf

# Fetch stock data from Yahoo Finance

def get_stock_price(ticker,history=5):
    # time.sleep(4) #To avoid rate limit error
    ticker = ticker.upper().strip()
    stock = yf.Ticker(ticker)
    df = stock.history(period="1y")
    df=df[["Close","Volume"]]
    df.index=[str(x).split()[0] for x in list(df.index)]
    df.index.rename("Date",inplace=True)
    df=df[-history:]
    # print(df.columns)
    
    return df.to_string()

# Fetch financial statements from Yahoo Finance
def get_financial_statements(ticker):
    # time.sleep(4) #To avoid rate limit error
    ticker = ticker.upper().strip()    
    company = yf.Ticker(ticker)
    balance_sheet = company.balance_sheet
    if balance_sheet.shape[1]>=3:
        balance_sheet=balance_sheet.iloc[:,:3]    # Remove 4th years data
    balance_sheet=balance_sheet.dropna(how="any")
    balance_sheet = balance_sheet.to_string()
    
    # cash_flow = company.cash_flow.to_string()
    # print(balance_sheet)
    # print(cash_flow)
    return balance_sheet



def analyze_week(df_weekly: pd.DataFrame, analysis_week_end=None, years_back=20):
    """
    Professional weekly analysis:
    - Gets current week index
    - Filters last N years
    - Selects only same Week_Number for seasonal analysis
    - Returns structured report + probability of next-week increase
    """

    # 1. Determine the week to analyze
    if analysis_week_end is None:
        analysis_week_end = df_weekly.index.max().date()

    analysis_week_end = pd.to_datetime(analysis_week_end)

    # Current Friday (week index)
    week_idx = df_weekly.index[df_weekly.index <= analysis_week_end].max()
    current = df_weekly.loc[week_idx]

    # Week number
    week_num = int(current["Week_Number"])

    # 2. Historical window
    window_start = week_idx - pd.DateOffset(years=years_back)
    hist_window = df_weekly.loc[window_start:week_idx]

    # 3. Filter identical week numbers
    hist = hist_window[hist_window["Week_Number"] == week_num].copy()
    # print(hist)
    # 4. Compute statistics
    mean_ret = hist["Weekly_return"].mean()
    std_ret = hist["Weekly_return"].std()
    min_ret = hist["Weekly_return"].min()
    max_ret = hist["Weekly_return"].max()

    # 5. Percentile of current week
    current_ret = current["Weekly_return"]
    percentile = (hist["Weekly_return"] <= current_ret).mean()

    # 🔥 6. Probability that NEXT WEEK goes UP (Target = 1)
    prob_up = hist["Target"].mean()

    # 7. Build professional report dictionary
    report = {
        "week_index": week_idx,
        "week_number": week_num,
        "years_back": years_back,
        "current_week": {
            "close": float(current["Close"]),
            "weekly_return": float(current["Weekly_return"]),
            "MACD": float(current["MACD"]),
            "MACD_signal": float(current["MACD_signal"]),
            "RSI_14": float(current["RSI_14"]),
            "Pandemic": int(current["Pandemic"])
        },
        "historical_stats": {
            "num_samples": int(len(hist)),
            "mean_weekly_return": float(mean_ret),
            "std_weekly_return": float(std_ret),
            "min_weekly_return": float(min_ret),
            "max_weekly_return": float(max_ret),
            "percentile_of_current_return": float(percentile)
        },
        "forecast": {
            "probability_next_week_up": float(prob_up),
            "probability_next_week_down": float(1 - prob_up)
        },
        "historical_dataframe": hist
    }

    return report,hist


# print(get_financial_statements("TATAPOWER.NS"))

# Ejemplo: analizar la semana más reciente
# hist_10y, current_week = analyze_week_vs_last_10_years(df_weekly)


In [12]:
df_daily_ta = add_technical_indicators(df.copy())
df_weekly = make_weekly_dataset(df_daily_ta)


print(df_weekly)


print(df_weekly.columns)
print(df_daily_ta.columns)
print(df_weekly.columns)



FEATURES = ["Weekly_return", "MACD", "MACD_signal", "RSI_14", "Pandemic"]

X = df_weekly[FEATURES]
y = df_weekly["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, shuffle=False, test_size=0.2
)

clf = LogisticRegression()
clf.fit(X_train, y_train)

# print(classification_report(y_test, clf.predict(X_test)))


def predict_next_week_direction(df_weekly, clf, features=FEATURES):
    last_row = df_weekly.iloc[[-1]]
    prob_up = clf.predict_proba(last_row[features])[0, 1]
    direction = "UP" if prob_up >= 0.5 else "DOWN"
    return direction, float(prob_up), last_row.index[0].date()

direction_hist, prob_up_hist, last_week_end = predict_next_week_direction(df_weekly, clf)
# print(f"Histórico → semana posterior a {last_week_end}: {direction_hist}, prob subir = {prob_up_hist:.3f}")


               Open     High      Low    Close       MACD  MACD_signal  \
Date                                                                     
2000-01-28  1441.40  1454.20  1356.10  1360.20 -14.440792    -7.196541   
2000-02-04  1360.20  1436.00  1350.00  1424.40  -9.460417   -10.410849   
2000-02-11  1424.40  1444.40  1379.20  1387.10  -7.563147    -7.737411   
2000-02-18  1387.10  1407.80  1345.30  1346.10 -14.323163   -10.002726   
2000-02-25  1346.10  1370.10  1329.10  1333.40 -20.898756   -15.206626   
...             ...      ...      ...      ...        ...          ...   
2025-10-24  6690.05  6807.11  6655.69  6791.69  38.354764    35.967525   
2025-10-31  6845.46  6920.34  6814.26  6840.20  62.295360    52.324275   
2025-11-07  6882.32  6882.32  6631.44  6728.80  33.391319    47.655962   
2025-11-14  6785.36  6869.91  6646.87  6734.11  24.032544    37.159065   
2025-11-21  6713.61  6754.50  6574.32  6617.37   0.393479    25.992506   

               RSI_14  Weekly_return 

In [13]:
#code for get lll sentimen response
## Source - https://stackoverflow.com/a/77998581
# Posted by Nikita Malviya, modified by community. See post 'Timeline' for change history
# Retrieved 2025-11-19, License - CC BY-SA 4.0
#pip install langchain-community langchain-core
%load_ext dotenv
%dotenv
NEWS_SOURCES = [
    "https://www.reuters.com/markets/us/",
    "https://www.cnbc.com/us-markets/",
    "https://www.bloomberg.com/markets",
    "https://finviz.com/news.ashx",
    "https://coinmarketcap.com/community/es/",
    "https://stockstory.org/",
    "https://www.tipranks.com/news"
]

import os
import time
from bs4 import BeautifulSoup
import re
import requests
import warnings

import os

from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain_core.tools import Tool  # for Tool()
from langchain_core.messages import SystemMessage, HumanMessage
import langchain, langchain_core
print("langchain:", langchain.__version__, langchain.__file__)
print("langchain_core:", langchain_core.__version__)
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import Tool
langchain.verbose = False
langchain.debug = False

import json

warnings.filterwarnings("ignore")





llm = ChatOpenAI(
    temperature=1,
    model_name="gpt-5-nano-2025-08-07",
)


# Script to scrap top5 googgle news for given company name

def google_query(search_term):
    if "news" not in search_term:
        search_term=search_term+" stock news"
    url=f"https://www.google.com/search?q={search_term}&cr=countryIN"
    url=re.sub(r"\s","+",url)
    return url

def get_recent_stock_news(company_name):
    # time.sleep(4) #To avoid rate limit error
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'}

    g_query=google_query(company_name)
    res=requests.get(g_query,headers=headers).text
    soup=BeautifulSoup(res,"html.parser")
    news=[]
    for n in soup.find_all("div","n0jPhd ynAwRc tNxQIb nDgy9d"):
        news.append(n.text)
    for n in soup.find_all("div","IJl0Z"):
        news.append(n.text)


    if len(news)>6:
        news=news[:4]
    else:
        news=news
    news_string=""
    for i,n in enumerate(news):
        news_string+=f"{i}. {n}\n"
    top5_news="Recent News:\n\n"+news_string
    
    return top5_news

def fetch_headlines(sources=NEWS_SOURCES, max_per_site=10):
    headlines = []
    for url in sources:
        try:
            res = requests.get(
                url,
                timeout=10,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            soup = BeautifulSoup(res.text, "html.parser")
            # muy genérico: h1, h2, h3
            count = 0
            for tag in soup.find_all(["h1", "h2", "h3"]):
                text = tag.get_text(strip=True)
                if len(text) > 25:
                    headlines.append(f"{url} :: {text}")
                    count += 1
                    if count >= max_per_site:
                        break
        except Exception as e:
            print(f"Error leyendo {url}: {e}")
    return headlines

search=DuckDuckGoSearchRun()
# Making tool list

tools=[
    Tool(
        name="get stock data",
        func=get_stock_price,
        description="Use when you are asked to evaluate or analyze a stock. This will output historic share price data. You should input the the stock ticker to it "
    ),
    Tool(
        name="DuckDuckGo Search",
        func=search.run,
        description="Use only when you need to get NSE/BSE stock ticker from internet, you can also get recent stock related news. Dont use it for any other analysis or task"
    ),
    Tool(
        name="get recent news",
        func=get_recent_stock_news,
        description="Use this to fetch recent news about stocks"
    ),

    Tool(
        name="get financial statements",
        func=get_financial_statements,
        description="Use this to get financial statement of the company. With the help of this data companys historic performance can be evaluaated. You should input stock ticker to it"
    ) 


]

''' function=[
        {
        "name": "get_company_Stock_ticker",
        "description": "This will get the indian NSE/BSE stock ticker of the company",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker_symbol": {
                    "type": "string",
                    "description": "This is the stock symbol of the company.",
                },

                "company_name": {
                    "type": "string",
                    "description": "This is the name of the company given in query",
                }
            },
            "required": ["company_name","ticker_symbol"],
        },
    }
] '''

def get_stock_ticker(query):
    function_def = {
        "name": "get_company_Stock_ticker",
        "description": "Extract the company name and stock ticker.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "Company name from the query.",
                },
                "ticker_symbol": {
                    "type": "string",
                    "description": "Extracted stock ticker symbol.",
                },
            },
            "required": ["company_name", "ticker_symbol"],
        },
    }

    response = llm.invoke(
        
        input=query,
        functions=[function_def],
        function_call={"name": "get_company_Stock_ticker"}
    )

    # Extract function call arguments
    func_args = response.additional_kwargs["function_call"]["arguments"]
    parsed = json.loads(func_args)

    return parsed["company_name"], parsed["ticker_symbol"]



def Anazlyze_stock(query):
    #agent.run(query) Outputs Company name, Ticker
    Company_name,ticker=get_stock_ticker(query)
    print({"Query":query,"Company_name":Company_name,"Ticker":ticker})
    stock_data=get_stock_price(ticker,history=10)
    stock_financials=get_financial_statements(ticker)
    stock_news=get_recent_stock_news(Company_name)

    # available_information=f"Stock Price: {stock_data}\n\nStock Financials: {stock_financials}\n\nStock News: {stock_news}"
    available_information=f"Stock Financials: {stock_financials}\n\nStock News: {stock_news}"

    print("\n\nAnalyzing.....\n")
    analysis=llm(f"Give detail stock analysis, Use the available data and provide investment recommendation. \
             The user is fully aware about the investment risk, dont include any kind of warning like 'It is recommended to conduct further research and analysis or consult with a financial advisor before making an investment decision' in the answer \
             User question: {query} \
             You have the following information available about {Company_name}. Write (5-8) pointwise investment analysis to answer user query, At the end conclude with proper explaination.Try to Give positives and negatives  : \
              {available_information} "
             )
    print(analysis)

    return analysis

import json

def summarize_market_sentiment(headlines, week_end_date):
    """
    Devuelve un dict:
      { "label": "bullish|bearish|neutral", "score": float, "explanation": str }
    """
    if not headlines:
        return {
            "label": "neutral",
            "score": 0.0,
            "explanation": "No headlines fetched."
        }

    text_block = "\n".join(f"- {h}" for h in headlines[:20])

    prompt = f"""
You are a professional equity analyst.

I will give you recent news headlines related to the US stock market.
Your task:

1. Classify overall sentiment for the S&P 500 for the week ending on {week_end_date} as one of:
   - bullish
   - bearish
   - neutral
2. Give a confidence score between 0 and 1.
3. Briefly explain the main reasons.

Headlines:
{text_block}

Return JSON ONLY in this format:
{{
  "label": "bullish",
  "score": 0.78,
  "explanation": "..."
}}
"""

    resp = llm.invoke(prompt)
    content = resp.content.strip()

    try:
        data = json.loads(content)
    except Exception:
        data = {
            "label": "neutral",
            "score": 0.0,
            "explanation": content[:400],
        }
    return data



The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
langchain: 0.2.10 d:\Python310\lib\site-packages\langchain\__init__.py
langchain_core: 0.2.43


In [14]:
news_headlines = fetch_headlines()
len(news_headlines), news_headlines[:5]

print("-----------------------------------")
print(news_headlines)
print("-----------------------------------")
print(get_recent_stock_news("spx"))
print("-----------------------------------")
search("Stock news USA")
print("-----------------------------------")

from datetime import date

today = date.today()  # p.ej. 2025-11-21
sentiment = summarize_market_sentiment(news_headlines, today)
print(sentiment)

-----------------------------------
['https://www.bloomberg.com/markets :: Nvidia, Alphabet, Ross Stores', 'https://www.bloomberg.com/markets :: Nvidia, Alphabet, Ross Stores', 'https://finviz.com/news.ashx :: Upgrade your FINVIZ experience', 'https://stockstory.org/ :: Ingram Micro (INGM) Stock Is Up, What You Need To Know', 'https://stockstory.org/ :: Oscar Health and Artivion Shares Are Soaring, What You Need To Know', 'https://stockstory.org/ :: EchoStar, Lumen, Exponent, TD SYNNEX, and IBM Shares Skyrocket, What You Need To Know']
-----------------------------------
Recent News:


-----------------------------------


d:\Python310\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
d:\Python310\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


-----------------------------------
{'label': 'bullish', 'score': 0.74, 'explanation': "Headlines highlight strong price action in mega-cap tech (Nvidia, Alphabet) and consumer names (Ross Stores), with multiple articles noting stocks 'up', 'soaring', or 'skyrocket'. This breadth of positive moves among influential S&P 500 components suggests constructive risk sentiment and upside potential for the week."}


# analisys step by step

In [15]:
# out = Analyze_stock("spx stock price prediction for this week and next week")

from pprint import pprint
import json

# =========================================
# 1) WEEKLY ANALYSIS REPORT
# =========================================
week_end_date = df_weekly.index.max().date()
report, hist = analyze_week(df_weekly)

print("\n===== WEEKLY ANALYSIS REPORT (JSON) =====")
print(json.dumps(report, indent=2, default=str))

print("\n==============================")
print(" CURRENT WEEK ANALYSIS")
print("==============================")
print(f"Week ending on: {week_end_date}")
print("==============================\n")


# =========================================
# 2) LLM NEWS SENTIMENT ANALYSIS
# =========================================
news_headlines = fetch_headlines()

sentiment = summarize_market_sentiment(news_headlines, week_end_date)

print("\n=== NEWS SENTIMENT (LLM BASED ON MARKET HEADLINES) ===")
print(json.dumps(sentiment, indent=2, default=str))


# =========================================
# 3) HISTORICAL MODEL (WEEKLY FORECAST)
# =========================================
probability_next_week_up = report["forecast"]["probability_next_week_up"] > 0.5


# =========================================
# 4) COMBINED INTERPRETATION
# =========================================
sentiment_label = sentiment.get("label", "neutral").lower()
sentiment_score = float(sentiment.get("score", 0.0))
explanation = sentiment.get("explanation", "")

print("\n=== COMBINED INTERPRETATION ===")

if direction_hist == "UP" and "bull" in sentiment_label:
    print(" Historical data and news sentiment align → **Bullish scenario**.")
elif direction_hist == "DOWN" and "bear" in sentiment_label:
    print("Historical data and news sentiment align → **Bearish scenario**.")
else:
    print(" Mixed signals → **Uncertain scenario**.")

print("\nLLM Explanation:")
print(explanation[:600])



===== WEEKLY ANALYSIS REPORT (JSON) =====
{
  "week_index": "2025-11-21 00:00:00",
  "week_number": 47,
  "years_back": 20,
  "current_week": {
    "close": 6617.37,
    "weekly_return": -0.017335624158203555,
    "MACD": 0.3934794795814014,
    "MACD_signal": 25.992505666114123,
    "RSI_14": 28.759562161825983,
    "Pandemic": 0
  },
  "historical_stats": {
    "num_samples": 21,
    "mean_weekly_return": -0.0036244698573535774,
    "std_weekly_return": 0.027555219650317292,
    "min_weekly_return": -0.0839345013168441,
    "max_weekly_return": 0.036252665637179105,
    "percentile_of_current_return": 0.23809523809523808
  },
  "forecast": {
    "probability_next_week_up": 0.7619047619047619,
    "probability_next_week_down": 0.23809523809523814
  },
  "historical_dataframe": "               Open     High      Low    Close       MACD  MACD_signal  \\\nDate                                                                     \n2005-11-25  1248.30  1270.60  1246.90  1268.20  15.376425 